<h1>Ideias e BrainStorm de Desenvolvimento</h1>

* Oque devemos olhar
* Quais validações são importantes para o cliente

In [1]:
import polars as pl

In [3]:
bases = '../bases/'

cartoes = pl.read_csv(f'{bases}cartoes.csv')
commits = pl.read_csv(f'{bases}commits.csv')
grupos = pl.read_csv(f'{bases}grupos.csv')
kanban_eventos = pl.read_csv(f'{bases}kanban_eventos.csv')
merge_requests = pl.read_csv(f'{bases}merge_requests.csv')
pessoas = pl.read_csv(f'{bases}pessoas.csv')
quadro_colunas = pl.read_csv(f'{bases}quadro_colunas.csv')
sprints = pl.read_csv(f'{bases}sprints.csv')

In [13]:
commits.with_columns(
    pl.col('commitado_em').str.slice(0,10).str.strptime( dtype=pl.Datetime, format="%Y-%m-%d")
)

grupo,commit_id,autor_id,autorado_em,commitado_em,e_merge,linhas_adicionadas,linhas_removidas,linhas_total,titulo,mensagem
str,str,str,str,datetime[μs],i64,i64,i64,i64,str,str
"""G01""","""f0c3a54e""","""[bot]""","""2022-04-08T14:56:14.000-03:00""",2022-04-08 00:00:00,0,3158,0,3158,"""Initial commit""","""Initial commit"""
"""G01""","""fb3b1012""","""[externo]""","""2022-04-08T15:38:59.000-03:00""",2022-04-08 00:00:00,0,0,2877,2877,"""Primeiro Upload""","""Primeiro Upload Primeira carg…"
"""G01""","""45b7e62f""","""[bot]""","""2022-04-17T23:23:43.000-03:00""",2022-04-17 00:00:00,0,104,0,104,"""Adicionando GitIgnore""","""Adicionando GitIgnore"""
"""G01""","""ec653018""","""[bot]""","""2022-04-17T23:31:52.000-03:00""",2022-04-17 00:00:00,0,5,3,8,"""Update README.md""","""Update README.md"""
"""G01""","""ba601797""","""[bot]""","""2022-04-28T10:00:34.000-03:00""",2022-04-28 00:00:00,0,2,0,2,"""Update README.md""","""Update README.md"""
…,…,…,…,…,…,…,…,…,…,…
"""G03""","""a8b0ca30""","""G03-A12""","""2026-06-26T17:09:56.000-03:00""",2026-06-26 00:00:00,0,19,19,38,"""fix(#445): marca campos NOT NU…","""fix(#445): marca campos NOT NU…"
"""G03""","""ad3dda87""","""G03-A12""","""2026-06-26T17:13:39.000-03:00""",2026-06-26 00:00:00,0,69,42,111,"""fix(#445): aplica obrigatorios…","""fix(#445): aplica obrigatorios…"
"""G03""","""0352cf09""","""G03-A12""","""2026-06-26T18:39:34.000-03:00""",2026-06-26 00:00:00,0,1,0,1,"""docs(#445): adiciona PUT /api/…","""docs(#445): adiciona PUT /api/…"


In [ ]:
kanban_eventos

In [ ]:
kanban = kanban_eventos.with_columns(
    pl.concat_str(pl.col('cartao_numero').cast(pl.String),pl.col('grupo')).alias('chave')
).join(
    cartoes.with_columns(
        pl.concat_str(pl.col('cartao_numero').cast(pl.String),pl.col('grupo')).alias('chave')
    ),
    on=pl.col('chave'),
    how='left'
)

kanban.write_parquet('/Users/PedroFranca/Desktop/Projeto dashboard Apartamento/pasta-git-projeto-data-viz/data_viz_projeto_pbl_final.grupo-davi_pena-pedro_franca-ulisses_pena/parquets/kanban.parquet')

In [ ]:
kanban.group_by(
    pl.col('grupo'),
    pl.col('sprint')
).agg(
    pl.len().alias('quantidade cards')
).with_columns(
    pl.coalesce(pl.col('sprint'),pl.lit('Sprint Não Informado')).alias('sprint')
)

In [ ]:
merge_requests.filter(
    pl.col('autor_id') == '[externo]'
).with_columns(
    pl.when(
        pl.col('autor_id') == '[externo]'
    ).then(
        pl.coalesce(pl.col('merged_por_id'),pl.col('revisores_ids'))
    ).alias('tratamento_responsavel_id')
)